# 04 — Feature Engineering & Preprocessing
**Project:** Transactional Fraud Detection Analysis  
**Objective:** Construct domain-relevant financial and temporal features, partition data using stratified splitting, and scale features with strict zero-leakage guarantees.



In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import DataLoader
from src.data_cleaning import DataCleaner
from src.feature_engineering import FeatureEngineer
from src.preprocessing import DataPreprocessor

loader = DataLoader()
df = loader.load_data()
cleaner = DataCleaner()
df_clean, _ = cleaner.clean_data(df)
print(f"Input records: {len(df_clean):,}")



## 1. Stratified Train-Test Partitioning (80/20)
We split data into 80% train and 20% test partitions using stratified sampling to preserve the exact ~0.17% fraud representation.



In [ ]:
preprocessor = DataPreprocessor()
X_train_raw, X_test_raw, y_train, y_test = preprocessor.split_data(
    df_clean,
    target_col="Class",
    test_size=0.20,
    random_state=42
)

print(f"Train Shape: {X_train_raw.shape}, Frauds: {y_train.sum():,} ({y_train.mean()*100:.3f}%)")
print(f"Test Shape:  {X_test_raw.shape}, Frauds: {y_test.sum():,} ({y_test.mean()*100:.3f}%)")



## 2. Feature Engineering Pipeline (Zero-Leakage)
We engineer `log_amount`, `amount_zscore`, `hour_of_day`, `hour_sin`, `hour_cos`, `is_night_transaction`, `v14_v17_ratio`, and `v10_v12_sum`.



In [ ]:
fe = FeatureEngineer()
X_train_fe = fe.fit_transform(X_train_raw)
X_test_fe = fe.transform(X_test_raw)

print(f"Features created: {X_train_fe.shape[1]} columns")
X_train_fe[["Amount", "log_amount", "amount_zscore", "hour_of_day", "is_night_transaction", "v14_v17_ratio"]].head()



## 3. Robust Scaling
We apply `RobustScaler` to skewed features, fitting parameters strictly on training features.



In [ ]:
X_train_scaled = preprocessor.fit_transform(X_train_fe)
X_test_scaled = preprocessor.transform(X_test_fe)

print("Preprocessing complete with zero data leakage.")

